In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dl_translate as dlt
import spacy
import tqdm
import time
from scipy import sparse
from googletrans import Translator as GoogleTranslator


AttributeError: module 'httpcore' has no attribute 'SyncHTTPTransport'

### Pré traitement des données 

In [2]:
df = pd.read_csv('../data/projects_data.csv')

In [3]:
df.head()

,project_name,link,description,composants
0,Tillu - the Robot,https://www.instructables.com/Tillu-the-Robot,"MeetTillu - The Robot, a unique fusion of adva...",1xUnihiker 4xServo 1xBattery Managment 2xUSB T...
1,Air Hockey Robot With Shot Prediction,https://www.instructables.com/Air-Hockey-Robot,Ever wanted to play air hockey against a compu...,Disclaimer - Supplies list may include affilia...
2,Two-Wheeled Drive Car With a Robotic Arm,https://www.instructables.com/Two-Wheeled-Driv...,This project was developed within theOrange Di...,To start building a Two-Wheeled Drive Car with...
3,Ghost Candy Dispenser,https://www.instructables.com/Ghost-Candy-Disp...,Hi there! Did you enjoy Halloween in 2024? Sin...,Parts MCU:M5StampS3 (M5Stack)(×1) RC servo mot...
4,Lily∞Bot With Micro:Bit and Motor:Bit Does Obs...,https://www.instructables.com/LilyBot-With-Mic...,This project will explain how to get obstacle ...,The parts list for the Lily∞Bot With Micro:Bit...


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   project_name  1499 non-null   object
 1   link          1503 non-null   object
 2   description   1178 non-null   object
 3   composants    1169 non-null   object
dtypes: object(4)
memory usage: 47.1+ KB


In [5]:
df.isna().sum()

project_name      4
link              0
description     325
composants      334
dtype: int64

On a:
- *325* projets sans descriptions
- *334* projets dont les composants pour la concéption ne sont pas indiqués.

Je vais tous simplement supprimer les lignes sans l'un de ces attributs car on ne pourra pas récommender à une personne un projet si on ne sait même pas de quoi parle le projet...

In [6]:
df_cleaned = df.dropna(subset=['composants','description'])
df_cleaned.isna().sum()

project_name    0
link            0
description     0
composants      0
dtype: int64

**Je vais traduire les textes en Français pour la suite de l'analyse**

In [38]:
def translate_text(text, retry_count=3):
    translator = GoogleTranslator()
    try:
        return translator.translate(text, src='en', dest='fr').text
    except:
        time.sleep(1)
        return text

def translate_column(df, column_name, batch_size=50):
    """Traduit une colonne par lots"""
    translated = []
    for i in tqdm.tqdm((range(0, len(df), batch_size))):
        batch = df[column_name].iloc[i:i+batch_size] 
        translated.extend([translate_text(text) for text in batch])
        time.sleep(2)  # Délai entre les lots
        
    return translated


In [ ]:
# Pour eviter les warning 
df_final = df_cleaned.copy()
df_final.loc[:, 'description_fr'] = translate_column(df_final, 'description')
df_final.loc[:, 'project_name_fr'] = translate_column(df_final, 'project_name')
df_final.loc[:, 'composants_fr'] = translate_column(df_final, 'composants')

In [3]:
df_final = pd.read_csv('../data/projects_translated_fr.csv')

In [4]:
df_final.head()

,link,description_fr,project_name_fr,composants_fr,description_preprocessed
0,https://www.instructables.com/tillu-the-robot,"meettillu - the robot, une fusion unique de ro...",tillu - le robot,1xunihiker 4xservo 1xbattery management 2xusb ...,meettillu the robot fusion unique robotique av...
1,https://www.instructables.com/air-hockey-robot,vous avez toujours voulu jouer au air hockey c...,robot de hockey sur air avec prédiction de tir,avis de non-responsabilité – la liste des four...,avoir vouloir jouer air hockey contre ordinate...
2,https://www.instructables.com/two-wheeled-driv...,ce projet a été développé au sein de l'orange ...,voiture à deux roues motrices avec bras robotique,pour commencer à construire une voiture à deux...,projet être développer sein orange digital cen...
3,https://www.instructables.com/ghost-candy-disp...,salut! avez-vous apprécié halloween en 2024 ? ...,distributeur de bonbons fantômes,pièces mcu : m5stamps3 (m5stack)(×1) servomote...,salut avoir vous apprécier halloween 2024 géné...
4,https://www.instructables.com/lilybot-with-mic...,ce projet expliquera comment éviter les obstac...,lily∞bot avec micro:bit et motor:bit évite les...,la liste des pièces du lily∞bot with micro:bit...,projet expliquer éviter obstacle robot modulai...


In [9]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1123 entries, 0 to 1123
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   link                      1123 non-null   object
 1   description_fr            1123 non-null   object
 2   project_name_fr           1123 non-null   object
 3   composants_fr             1123 non-null   object
 4   description_preprocessed  1123 non-null   object
dtypes: object(5)
memory usage: 52.6+ KB


On se retrouve avec un seul projet sans composants, je vais la supprimer par la suite

In [6]:
df_final.dropna(inplace=True)

**Suppression des colonnes de base et conversion des textes en minuscules**

In [7]:
df_final = df_final.drop(columns=['project_name','description','composants'])

KeyError: "['project_name', 'description', 'composants'] not found in axis"

In [8]:
for col in df_final.columns:
    df_final[col] = df_final[col].str.lower()

**Tokenisation et lemmatisation (peut-être) de la colonne description**

In [13]:
nlp = spacy.load("fr_core_news_md")

# Traiter les textes en lot
def preprocess_texts_batch(texts, batch_size=100):
    docs = nlp.pipe(texts, batch_size=batch_size, disable=["ner", "parser"])
    processed_texts = [
        " ".join(token.lemma_ for token in doc if not token.is_stop and not token.is_punct)
        for doc in docs
    ]
    return processed_texts

In [14]:
df_final['description_preprocessed'] = preprocess_texts_batch(df_final['description_fr'])

In [20]:
# Sauvegarde le dataframe final
df_final.to_csv('../data/projects_translated_fr.csv',index=False)

### Préparation du système de recommandation

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
# Initialisation du vectoriseur TF-IDF
vectorizer = TfidfVectorizer()
# On va appliquer TF-IDF sur les descriptions prétraitées
tfidf_matrix = vectorizer.fit_transform(df_final['description_preprocessed'])

In [21]:
import pickle

with open("../src/features/tfidf_model_project.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [15]:
sparse.save_npz('../src/features/tfidf_description_project_matrix.npz',tfidf_matrix)

In [16]:
# Fonction pour calculer les similarités
def find_similar_descriptions(user_input, top_n=5):
    """
    Trouver les descriptions les plus similaires à partir de l'entrée utilisateur.
    Paramètres :
        user_input (str) : Description donnée par l'utilisateur.
        top_n (int) : Nombre de résultats les plus similaires à retourner.
    Retourne :
        DataFrame avec les descriptions et scores de similarité.
    """
    # pré-traitement de l'entrée de l'utilisateur
    user_vec = vectorizer.transform([user_input])
    # similarités cosinus 
    similarities = cosine_similarity(user_vec, tfidf_matrix).flatten()
    # les indices des descriptions les plus similaires
    top_indices = similarities.argsort()[::-1][:top_n]
    # enfin, on retourne les descriptions correspondantes avec les scores
    results = pd.DataFrame({
        'description': df_final.loc[top_indices, 'description_fr'],
        'similarity_score': similarities[top_indices],
        'nom_projet': df_final.loc[top_indices, 'project_name_fr'],
        'composants': df_final.loc[top_indices, 'composants_fr']
    })
    return results


In [17]:
user_description = "Je veux construire un robot suiveur de ligne"
result = find_similar_descriptions(user_description, top_n=3)

In [18]:
result

,description,similarity_score,nom_projet,composants
181,les robots suiveurs de ligne sont l'un des pro...,0.280141,robot suiveur de ligne à 4 roues motrices,l’idée de base est de faire en sorte que la co...
95,le robot suiveur de ligne est une étape fondam...,0.247922,suiveur de ligne avec contrôleur de robot esp3...,kit robot suiveur de ligne - c'est le kit à 5 ...
915,les servos sont de petits boîtiers qui contien...,0.243808,pirater un servo,voici une liste des pièces et des outils que j...
